# Module 5 — Inverse Kinematics
## Unit 3 — Analytical (Closed-Form) Inverse Kinematics
### Lesson 3.2 — The atan2 Tool and Choosing the Right Quadrant

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*


## Demonstration (§7)
atan2 keeps the quadrant; arctan does not.


In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

def fk_two_link(theta1, theta2, L1, L2):
    x = L1*np.cos(theta1) + L2*np.cos(theta1+theta2)
    y = L1*np.sin(theta1) + L2*np.sin(theta1+theta2)
    return np.array([x, y])

def ik_2link_closed(x, y, L1, L2):
    """Closed-form: return both (theta1,theta2); [] unreachable; one on boundary."""
    c2 = (x*x + y*y - L1*L1 - L2*L2)/(2*L1*L2)
    if c2 < -1-1e-9 or c2 > 1+1e-9:
        return []
    c2 = max(-1.0, min(1.0, c2))
    sols=[]
    for sign in (+1.0, -1.0):
        s2 = sign*np.sqrt(max(0.0, 1.0 - c2*c2))
        t2 = np.arctan2(s2, c2)
        t1 = np.arctan2(y, x) - np.arctan2(L2*np.sin(t2), L1 + L2*np.cos(t2))
        sols.append((t1, t2))
        if abs(s2) < 1e-6:
            break
    return sols

def jacobian_2link(theta1, theta2, L1, L2):
    s1, s12 = np.sin(theta1), np.sin(theta1+theta2)
    c1, c12 = np.cos(theta1), np.cos(theta1+theta2)
    return np.array([[-L1*s1 - L2*s12, -L2*s12],
                     [ L1*c1 + L2*c12,  L2*c12]])

pts=[(0.5,0.5),(-0.5,0.5),(-0.5,-0.5),(0.0,0.5)]
print(f"{'target':<14}{'atan2(deg)':<12}{'arctan(deg)':<12}")
for (x,y) in pts:
    a2=np.degrees(np.arctan2(y,x))
    at=np.degrees(np.arctan(y/x)) if x!=0 else float('nan')
    print(f"{str((x,y)):<14}{a2:<12.1f}{at:<12.1f}")


## Coding Exercise (§8)
Show where the two disagree; confirm behind-the-base IK is correct.


In [ ]:
# YOUR CODE HERE
assert round(np.degrees(np.arctan2(0.5,-0.5)))==135    # 2nd quadrant
assert round(np.degrees(np.arctan2(-0.5,-0.5)))==-135  # 3rd quadrant
# behind-the-base target solves with correct-quadrant shoulder
for (t1,t2) in ik_2link_closed(-0.4,0.2,L1:=0.4,L2:=0.3):
    assert np.allclose(fk_two_link(t1,t2,L1,L2),[-0.4,0.2],atol=1e-9)
print("quadrant handling OK")


## Check your work


In [ ]:
assert round(np.degrees(np.arctan2(0.5,0.0)))==90   # straight up; arctan undefined
print("All checks passed.")
